# 서명 TF-IDF — 원문 토큰은 왜 test 에서 죽는가

test 는 같은 변이를 전사체마다 다른 잔기 번호로 반복 기재한다(`M267I M206I`). 그래서 `MUT__{gene}__{token}` 어휘를 train 으로 만들면 test 문서가 거의 안 걸린다. 잔기 번호를 지운 **서명** (`SIG__{gene}__missense|M>I`)으로 바꾸면 그 격차가 사라진다.

이 노트북이 재는 것

1. 문서 두 벌의 길이와 어휘 크기
2. `min_df` 별 train/test 커버리지 — 기본값 3 의 근거
3. fold-safe OOF 로 두 문서 비교
4. **누수 반증** — 일부러 train+test 를 합쳐 어휘를 만들면 점수가 얼마나 뜨는지

> 규정: IDF 는 문서 집합 통계다. 어휘·IDF·chi2 를 **fold 의 train 부분에만** fit 한다. train+test 를 합쳐 어휘를 만들면 실격이다.

In [ ]:
import sys
from pathlib import Path

# notebooks/ 에서 열든 저장소 루트에서 열든 같은 곳을 가리키게 한다.
ROOT = Path.cwd()
while not (ROOT / "src" / "cancer_hack").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "process"
print(ROOT)


In [ ]:
import numpy as np
import pandas as pd

from cancer_hack.features_sparse import MutationTfidfBlock
from cancer_hack.metrics import macro_f1

train_docs = pd.read_parquet(DATA_PROC / "train_signature_mutation_tokens.parquet")
test_docs = pd.read_parquet(DATA_PROC / "test_signature_mutation_tokens.parquet")
train_exact = pd.read_parquet(DATA_PROC / "train_exact_mutation_tokens.parquet")
test_exact = pd.read_parquet(DATA_PROC / "test_exact_mutation_tokens.parquet")

DOCS = {
    "exact": (
        train_exact["exact_mutation_document"].to_numpy(object),
        test_exact["exact_mutation_document"].to_numpy(object),
    ),
    "signature": (
        train_docs["unique_mutation_document"].to_numpy(object),
        test_docs["unique_mutation_document"].to_numpy(object),
    ),
}
y = train_docs["SUBCLASS"].to_numpy()
folds = pd.read_parquet(DATA_PROC / "train_folds.parquet")
assert (folds["ID"].to_numpy() == train_docs["ID"].to_numpy()).all()
fold_ids = folds["fold_skf5"].to_numpy()
print({k: (len(a), len(b)) for k, (a, b) in DOCS.items()})


## 1. 문서 길이 — 서명이 test 쪽 부풀림을 절반으로 줄인다

test 가 train 보다 3배 긴 건 이소폼 반복 때문이다. 서명으로 접으면 그 부분이 빠진다.

In [ ]:
rows = []
for kind, (tr, te) in DOCS.items():
    rows.append(
        {
            "문서": kind,
            "train 행당 토큰": np.mean([len(d.split()) for d in tr]).round(1),
            "test 행당 토큰": np.mean([len(d.split()) for d in te]).round(1),
            "train 고유 토큰": len({t for d in tr for t in d.split()}),
            "test 고유 토큰": len({t for d in te for t in d.split()}),
        }
    )
pd.DataFrame(rows).set_index("문서")


## 2. 어휘 진단 — `min_df` 를 3 으로 잡은 근거

train 에만 fit 하고 test 를 transform 한다. 보는 값은 **test 비영 행 비율**과 **행당 비영 항 수**다. 0 벡터가 많으면 그 블록은 test 에서 아무 말도 못 한다.

In [ ]:
diagnostics = []
for kind, (tr, te) in DOCS.items():
    for min_df in (1, 2, 3, 5, 10):
        block = MutationTfidfBlock(min_df=min_df, max_features=None).fit(tr)
        mtr, mte = block.transform(tr), block.transform(te)
        diagnostics.append(
            {
                "문서": kind,
                "min_df": min_df,
                "|V|": len(block.feature_names_),
                "train 비영 행": f"{(np.diff(mtr.indptr) > 0).mean():.1%}",
                "test 비영 행": f"{(np.diff(mte.indptr) > 0).mean():.1%}",
                "test 행당 nnz": round(float(np.diff(mte.indptr).mean()), 1),
            }
        )
diagnostic_frame = pd.DataFrame(diagnostics)
diagnostic_frame


**읽는 법.** 원문(`exact`)은 `min_df` 를 올릴수록 test 행당 비영 항이 1 근처로 주저앉는다 — 사실상 0 행렬이다. 서명은 train 과 test 의 비영 비율이 거의 붙어 있고, `min_df=3` 에서 그 차이가 가장 작으면서 어휘가 2 만 대로 떨어진다. 그래서 `MUTATION_TFIDF_DEFAULTS["min_df"] = 3` 이다.

In [ ]:
pivot = diagnostic_frame.pivot(index="min_df", columns="문서", values="test 행당 nnz")
ax = pivot.plot(marker="o", figsize=(6, 3.5), title="min_df 별 test 행당 비영 항")
ax.set_ylabel("nnz / row")
ax.grid(alpha=0.3)


## 3. fold-safe OOF 비교

fold 마다 어휘·IDF·chi2 를 train 부분에서만 다시 만든다. 모델은 노트북에서 빠르게 돌리려고 로지스틱 회귀를 쓴다 — 절대 점수가 아니라 두 문서의 **차이**를 보는 게 목적이다.

In [ ]:
from sklearn.linear_model import LogisticRegression

from cancer_hack.validation import Chi2TopKSelector

classes = np.unique(y)


def fold_safe_oof(documents, *, min_df=3, topk=1000, seed=42):
    """fold 마다 어휘를 새로 만든 OOF 예측. valid 는 transform 만 받는다."""
    oof = np.empty(len(y), dtype=object)
    picked_per_fold = []
    for fold in range(5):
        valid_index = np.where(fold_ids == fold)[0]
        train_index = np.where(fold_ids != fold)[0]

        block = MutationTfidfBlock(min_df=min_df, max_features=None)
        block.fit(documents[train_index])
        matrix = block.transform(documents)

        selector = Chi2TopKSelector(k=topk).fit(matrix[train_index], y[train_index])
        picked_per_fold.append(set(block.feature_names_[selector.indices_]))
        reduced = selector.transform(matrix)

        model = LogisticRegression(max_iter=2000, random_state=seed)
        model.fit(reduced[train_index], y[train_index])
        oof[valid_index] = model.predict(reduced[valid_index])
    return oof.astype(str), picked_per_fold


oof_scores = {}
picked = {}
for kind, (tr, _) in DOCS.items():
    predictions, picked[kind] = fold_safe_oof(tr)
    oof_scores[kind] = macro_f1(y, predictions)
    print(f"{kind:10s} OOF Macro F1 = {oof_scores[kind]:.4f}")


In [ ]:
singleton = folds.groupby("group_key")["ID"].transform("size").to_numpy() == 1
summary = pd.DataFrame(
    {
        "전체 OOF": pd.Series(oof_scores),
        "단독 행": pd.Series(
            {
                kind: macro_f1(y[singleton], fold_safe_oof(tr)[0][singleton])
                for kind, (tr, _) in DOCS.items()
            }
        ),
    }
)
summary.plot.bar(figsize=(6, 3.5), rot=0, title="문서 표현별 OOF Macro F1").grid(alpha=0.3)
summary


## 4. `sparse_topk` 스윕

chi2 로 몇 열까지 남길지. 열이 늘수록 좋아지다가 평평해지는 지점을 찾는다.

In [ ]:
sweep = []
signature_train = DOCS["signature"][0]
for topk in (250, 500, 1000, 2000):
    predictions, _ = fold_safe_oof(signature_train, topk=topk)
    sweep.append({"sparse_topk": topk, "OOF Macro F1": macro_f1(y, predictions)})
sweep_frame = pd.DataFrame(sweep).set_index("sparse_topk")
sweep_frame.plot(marker="o", figsize=(6, 3.5), title="chi2 top-K vs OOF").grid(alpha=0.3)
sweep_frame


## 5. fold 간 선택 안정성

fold 마다 다른 항이 뽑히면 그 블록은 잡음을 보고 있는 것이다. Jaccard 가 높아야 믿을 수 있다.

In [ ]:
def mean_jaccard(sets):
    values = []
    for i in range(len(sets)):
        for j in range(i + 1, len(sets)):
            values.append(len(sets[i] & sets[j]) / len(sets[i] | sets[j]))
    return float(np.mean(values))


for kind, sets in picked.items():
    print(f"{kind:10s} fold 간 평균 Jaccard = {mean_jaccard(sets):.3f}")

common = set.intersection(*picked["signature"])
print(f"\n5 fold 전부에서 뽑힌 서명 항 {len(common)}개 중 20개:")
for name in sorted(common)[:20]:
    print("  ", name)


## 6. 누수 반증 — 이렇게 하면 안 된다

일부러 train+test 를 합쳐 어휘와 IDF 를 만들어 본다. 규정 위반이므로**절대 제출 파이프라인에 넣지 않는다.** 여기서 재는 건 그 편향의 크기다.

In [ ]:
def leaky_oof(train_documents, test_documents, *, min_df=3, topk=1000):
    """train+test 를 합쳐 어휘를 만든다 — 규정 위반. 편향 크기 측정 전용이다."""
    block = MutationTfidfBlock(min_df=min_df, max_features=None)
    block.fit(np.concatenate([train_documents, test_documents]))
    matrix = block.transform(train_documents)

    oof = np.empty(len(y), dtype=object)
    for fold in range(5):
        valid_index = np.where(fold_ids == fold)[0]
        train_index = np.where(fold_ids != fold)[0]
        selector = Chi2TopKSelector(k=topk).fit(matrix[train_index], y[train_index])
        reduced = selector.transform(matrix)
        model = LogisticRegression(max_iter=2000, random_state=42)
        model.fit(reduced[train_index], y[train_index])
        oof[valid_index] = model.predict(reduced[valid_index])
    return oof.astype(str)


leaky = macro_f1(y, leaky_oof(*DOCS["signature"]))
honest = oof_scores["signature"]
print(f"fold-safe        {honest:.4f}")
print(f"train+test 합침  {leaky:.4f}   (차이 {leaky - honest:+.4f})")
print("\n어휘를 합치면 CV 가 이만큼 낙관적으로 나온다. 리더보드에는 그 보정이 없다.")


## 결론

- 서명 문서를 쓴다. 원문 토큰 TF-IDF 는 test 에서 0 행렬에 가깝다.
- `min_df=3`, `sparse_topk=1000` 을 기본값으로 둔다.
- 학습 스크립트에서 확인하기

```powershell
$env:PYTHONUTF8 = "1"
$PY = "D:\Code\Final_Hachathon\code\.venv\Scripts\python.exe"
& $PY scripts/train_gbdt.py --model xgb --configs f4r,f5,f5x --cv skf `
    --no-submission --tag dup
```

`f5`(서명)가 `f5x`(원문)보다 높아야 위 진단표와 앞뒤가 맞는다.

점수는 코드 주석이 아니라 노션 「모델 성능 기록」에 남긴다.